In [11]:
import random
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS

random_state = 2
#need both bc ones for py gen and other controls np
np.random.seed(random_state)
random.seed(random_state)

path = "data/all_book_titles.txt"

#got code from other labs
with open(path, encoding="utf-8") as f:
    raw_lines = f.read().splitlines()

titles = [t.strip() for t in raw_lines if t.strip()]

print("raw lines:", len(raw_lines))
print("non-empty titles:", len(titles))
#should make one column table
df = pd.DataFrame({"title": titles})
df.head(10)

uni = df["title"].nunique()
print("NEED TO BE REMOVED:", len(df) - uni)


#shows the stats of each title length
df["n_words"] = df["title"].str.split().str.len()
print(df["n_words"].describe())
#looked this up, splits titles into words, puts every word on its own row, counts them, output a table of 40 most common words
print(df["title"].str.lower().str.split().explode().value_counts().head(40))

#Found 142 duplicates want to remove them in the cleaning section. Titles have a median of 6 words (min 1, max 36). Noise words I plan to remove are the stop words, along with
#publishing words (edition, th, access), the "s" leftover from possesives, and other vague words (introduction, approach).


#cleaning
#drops dups
df = df.drop_duplicates(subset="title").reset_index(drop=True)
print("Should be 0:", len(df) - uni)

noise = ["edition", "th", "s", "access", "introduction", "approach",
         "5th", "volume", "series", "book", "brief", "essentials", "card", "real"]
stop_list = list(ENGLISH_STOP_WORDS) + noise

#skips stop words and just keeps words that are in at least 5 titles -- maybe switch later i just think rarer words will show no pattern whatsoever
c_v = CountVectorizer(stop_words=stop_list, min_df=5)

#makes the vocab from c_v and then builds table with on row per title and on column per word, each row tells us how many times that word appears
X_counts = c_v.fit_transform(df["title"])

#basically same thing except each cell is weighted. lower score = appear more / higher score = rarer words
tfidf_v = TfidfVectorizer(stop_words=stop_list, min_df=5)
X_tfidf = tfidf_v.fit_transform(df["title"])

print(len(c_v.get_feature_names_out()))

#LDA is given counts because it is a probability model. It treeats each title as if its words were drawn form topics, so it is built on how many times a word shows up.
#Raw counts fit perfectly with this model. While TF-IDF scores are scaleed dec that don't track word count. NMF only needs non-neg numbers, so we can use TF_IDF, which
#lowers the weight to words that appear more vis versa.

#Choose to remove dup words bc it would make words look more comman than they actually are inflating them
#using stop words bc they have nothing to do with topics
# added  "edition, th, access" to stop words bc it sjust publishing noise
#choose the min doc freq as 5 bc fewer is too rare to show any pattern at all





raw lines: 2373
non-empty titles: 2373
NEED TO BE REMOVED: 142
count    2373.000000
mean        7.142857
std         4.096361
min         1.000000
25%         4.000000
50%         6.000000
75%         9.000000
max        36.000000
Name: n_words, dtype: float64
title
and             932
of              576
the             489
edition)        321
to              320
a               304
in              304
for             258
introduction    236
with            180
an              175
edition         136
:               130
history         104
&               104
science         100
computer         88
approach         87
access           77
business         75
human            71
applications     68
principles       67
analysis         64
essentials       63
series           62
biology          61
art              58
theory           58
guide            56
fundamentals     56
statistics       53
card             53
chemistry        52
concepts         51
anatomy          51
design       

In [13]:
from sklearn.decomposition import LatentDirichletAllocation, NMF

n_topics = 3

#just creates the model
lda = LatentDirichletAllocation(n_components=n_topics, random_state=random_state)
#uses the x_counts talbe made above one row per title, one column per word, each cells tells how many time it appears in that title. Looks at which words co-occur across all the titles and figures
lda.fit(X_counts)
#basically same thing but nowa bigger number means that word matters more to a particular topic.
nmf = NMF(n_components=n_topics, random_state=random_state)
nmf.fit(X_tfidf)

,"n_components n_components: int or {'auto'} or None, default='auto'Number of components. If `None`, all features are kept.If `n_components='auto'`, the number of components is automatically inferredfrom W or H shapes... versionchanged:: 1.4 Added `'auto'` value... versionchanged:: 1.6 Default value changed from `None` to `'auto'`.",3
,"random_state random_state: int, RandomState instance or None, default=NoneUsed for initialisation (when ``init`` == 'nndsvdar' or'random'), and in Coordinate Descent. Pass an int for reproducibleresults across multiple function calls.See :term:`Glossary <random_state>`.",2
,"init init: {'random', 'nndsvd', 'nndsvda', 'nndsvdar', 'custom'}, default=NoneMethod used to initialize the procedure.Valid options:- `None`: 'nndsvda' if n_components <= min(n_samples, n_features), otherwise random.- `'random'`: non-negative random matrices, scaled with: `sqrt(X.mean() / n_components)`- `'nndsvd'`: Nonnegative Double Singular Value Decomposition (NNDSVD) initialization (better for sparseness)- `'nndsvda'`: NNDSVD with zeros filled with the average of X (better when sparsity is not desired)- `'nndsvdar'` NNDSVD with zeros filled with small random values (generally faster, less accurate alternative to NNDSVDa for when sparsity is not desired)- `'custom'`: Use custom matrices `W` and `H` which must both be provided... versionchanged:: 1.1 When `init=None` and n_components is less than n_samples and n_features defaults to `nndsvda` instead of `nndsvd`.",None
,"solver solver: {'cd', 'mu'}, default='cd'Numerical solver to use:- 'cd' is a Coordinate Descent solver.- 'mu' is a Multiplicative Update solver... versionadded:: 0.17 Coordinate Descent solver... versionadded:: 0.19 Multiplicative Update solver.",'cd'
,"beta_loss beta_loss: float or {'frobenius', 'kullback-leibler', 'itakura-saito'}, default='frobenius'Beta divergence to be minimized, measuring the distance between Xand the dot product WH. Note that values different from 'frobenius'(or 2) and 'kullback-leibler' (or 1) lead to significantly slowerfits. Note that for beta_loss <= 0 (or 'itakura-saito'), the inputmatrix X cannot contain zeros. Used only in 'mu' solver... versionadded:: 0.19",'frobenius'
,"tol tol: float, default=1e-4Tolerance of the stopping condition.",0.0001
,"max_iter max_iter: int, default=200Maximum number of iterations before timing out.",200
,"alpha_W alpha_W: float, default=0.0Constant that multiplies the regularization terms of `W`. Set it to zero(default) to have no regularization on `W`... versionadded:: 1.0",0.0
,"alpha_H alpha_H: float or ""same"", default=""same""Constant that multiplies the regularization terms of `H`. Set it to zero tohave no regularization on `H`. If ""same"" (default), it takes the same value as`alpha_W`... versionadded:: 1.0",'same'
,"l1_ratio l1_ratio: float, default=0.0The regularization mixing parameter, with 0 <= l1_ratio <= 1.For l1_ratio = 0 the penalty is an elementwise L2 penalty(aka Frobenius Norm).For l1_ratio = 1 it is an elementwise L1 penalty.For 0 < l1_ratio < 1, the penalty is a combination of L1 and L2... versionadded:: 0.17 Regularization parameter *l1_ratio* used in the Coordinate Descent solver.",0.0
,"verbose verbose: int, default=0Whether to be verbose.",0


In [14]:
#next need to get the top ten words for every topic in the model. 

def show_topics(model, vectorizer, n=10):
    words = vectorizer.get_feature_names_out()
    for i, comp in enumerate(model.components_):
        top = [words[j] for j in comp.argsort()[-n:][::-1]]
        print(f"Topic {i}: {', '.join(top)}")

print("LDA"); show_topics(lda, c_v)
print("\nNMF"); show_topics(nmf, tfidf_v)

LDA
Topic 0: analysis, applications, computer, world, theory, mechanics, design, data, health, probability
Topic 1: business, science, systems, economics, management, language, finance, calculus, sociology, marketing
Topic 2: history, biology, art, human, science, chemistry, physiology, anatomy, plus, psychology

NMF
Topic 0: history, art, world, american, america, film, modern, global, myartslab, western
Topic 1: science, computer, statistics, applications, business, theory, probability, analysis, design, earth
Topic 2: biology, human, physiology, molecular, anatomy, chemistry, life, concepts, plus, principles


In [18]:
#labels for part 4

nmf_labels = {
    0: "History, art and culture",              #history, art, american, film, modern, western
    1: "Computing, stats and applied science",  #computer, statistics, probability, analysis, design. business is mixed in so it's broad
    2: "Life sciences",                         #biology, physiology, molecular, anatomy, chemistry. the cleanest topic
}

lda_labels = {
    0: "Math, engineering and data",            #4 of 5 top titles are math/technical, medical dictionary is an outlier
    1: "Finance (plus 2 NLP conference titles)", #3 finance books, 2 NLP proceedings mixed in. repeated publisher text in long titles
    2: "Unclear",                               #unclear bc top 5 titles are all diff subjects (anatomy, art history, anthropology, geography, biology), only thing they share is the "Plus ... Access Card Package" publisher text
}

In [ ]:
#step 5: find the top 5 titles for each topic

#get each title's score for every topic
nmf_scores = nmf.transform(X_tfidf)
lda_scores = lda.transform(X_counts)

#NMF: go through topics 0, 1, 2
for topic in range(3):
    print("NMF topic", topic, "-", nmf_labels[topic])

    #get the scores for this one topic
    scores = nmf_scores[:, topic]

    #sort title positions from lowest to highest score, keep the last 5, then flip so highest is first
    sorted_positions = scores.argsort()
    top_5_positions = sorted_positions[-5:][::-1]

    #print each of those titles
    for position in top_5_positions:
        print("   ", df["title"].iloc[position])
    print()

#LDA: same thing
for topic in range(3):
    print("LDA topic", topic, "-", lda_labels[topic])

    scores = lda_scores[:, topic]

    sorted_positions = scores.argsort()
    top_5_positions = sorted_positions[-5:][::-1]

    for position in top_5_positions:
        print("   ", df["title"].iloc[position])
    print()
    
    #For steph 5, I chcke the five highest-scoring titles for three topics for each model. The NMF titles matched all three of my labels, th ehistory topic return art history books,
    #the computing topic returned computer science books, and the life sciences topic returned biology books. LDA didn't do aswell. Its math topic was motly differeintla equation, actuarial,
    #and network security books, with one medical dictionary that didn't fit. LDA works from raw word counts, so long titles that repeat the same words, like the repeated publisher 
    #series names, pull it toward them, while NMF’s TF-IDF weighting gave cleaner topics overall.

NMF topic 0 - History, art and culture
    Art History (5th Edition)
    History of Far Eastern Art
    Art History Volume 2 (5th Edition)
    Art History Portables Book 6 (5th Edition)
    Art History Portables Book 3 (5th Edition)

NMF topic 1 - Computing, stats and applied science
    Computer Science Illuminated
    Connecting with Computer Science
    Computer Science : An Overview
    Invitation to Computer Science
    Logic in Computer Science Modelling and Reasoning About Systems

NMF topic 2 - Life sciences
    Visualizing Human Biology
    Human Biology
    Molecular Biology of the Gene
    Molecular Biology: Genes to Proteins
    Bscs Biology A Molecular Approach

LDA topic 0 - Math, engineering and data
    Mosby's Dictionary of Medicine, Nursing & Health Professions (Mosby's Dictionary of Medicine, Nursing, and Health Professions)
    Differential Equations with Boundary Value Problems : An Introduction to Modern Methods and Applications
    Guide to Computer Network Secur

**Step 5 (top titles).** The NMF titles matched all three labels (art history books, computer science books, biology books). LDA matched less. Its math topic was mostly differential equations, actuarial and network security books with one medical dictionary that doesn't fit, and its finance topic had 2 NLP proceedings in it. LDA works from raw counts, so long titles that repeat the same words (like series names) pull it around, while NMF's TF-IDF weighting gave cleaner topics. Note that many titles tie on score (all the Art History editions score the same), so which five show up can change a little between runs on different computers.

In [21]:
#step 6: try 5, 10, and 15 topics
for k in [5, 10, 15]:
    print("k =", k)

    lda_k = LatentDirichletAllocation(n_components=k, random_state=random_state)
    lda_k.fit(X_counts)
    print("LDA")
    show_topics(lda_k, c_v)

    nmf_k = NMF(n_components=k, random_state=random_state)
    nmf_k.fit(X_tfidf)
    print("NMF")
    show_topics(nmf_k, tfidf_v)
    print("NMF error:", nmf_k.reconstruction_err_)
    print()
    
#For step 6, I tried 5, 10, and 15 topics. At 5, the topics were still mixed, and one LDA topic was mostly publisher filler like “plus” and “package”.
#At 10, the topics were clearer, with separate finance, religion, computing, and biology topics. At 15, some topics started to overlap, like the systems and management ones.
#NMF error only dropped from 45.53 to 43.27 and keeps falling as k grows, so I picked k = 10 based on how clear the topics were.

k = 5
LDA
Topic 0: mechanics, data, probability, statistics, health, principles, structures, engineers, security, medicine
Topic 1: business, computer, management, economics, sociology, systems, physics, learning, 3rd, social
Topic 2: history, art, physiology, anatomy, plus, package, literature, american, etext, human
Topic 3: science, theory, biology, applications, psychology, analysis, guide, design, world, engineering
Topic 4: language, finance, chemistry, concepts, anthropology, student, human, manual, natural, processing
NMF
Topic 0: history, art, american, world, america, film, modern, global, myartslab, western
Topic 1: science, computer, earth, political, organization, networks, physical, architecture, chemistry, forensic
Topic 2: biology, molecular, chemistry, concepts, life, campbell, biochemistry, cell, organic, general
Topic 3: business, statistics, economics, theory, analysis, probability, applications, principles, management, design
Topic 4: human, physiology, anatomy, se

**Step 6 (number of topics).** We tried 5, 10 and 15 topics. At 5 the topics were still mixed. At 10 they were clearer, with separate finance, religion, computing and biology topics. At 15 some topics started to overlap, like the systems and management ones. The NMF reconstruction error went from 45.53 (5 topics) to 44.35 (10) to 43.27 (15), but it keeps going down as the number of topics grows, so it can't pick the number for us. We didn't use LDA perplexity. We picked 10 because the topics were the clearest to read.

In [22]:
#step 7: topic mixtures for five titles (k = 10)
nmf10 = NMF(n_components=10, random_state=random_state)
nmf10.fit(X_tfidf)

#each title's score for each of the 10 topics
mix = nmf10.transform(X_tfidf)

#make each row add up to 1 so it reads like percentages
mix = mix / (mix.sum(axis=1, keepdims=True) + 0.000001)

#five titles to look at
for position in [0, 100, 500, 1000, 2000]:
    print(df["title"].iloc[position])
    print(mix[position].round(2))
    print()

Philosophy of Sex and Love A Reader
[0.2  0.   0.02 0.04 0.03 0.   0.05 0.1  0.44 0.11]

The Theater Experience
[0.36 0.   0.   0.   0.   0.   0.09 0.07 0.22 0.26]

The Convergence of Race, Ethnicity, and Gender: Multiple Identities in Counseling (4th Edition) (Merrill Counseling)
[0.15 0.   0.12 0.06 0.1  0.05 0.   0.   0.36 0.16]

Essentials of Human Anatomy & Physiology (10th Edition)
[0.   0.   0.   0.   0.94 0.   0.   0.   0.   0.06]

Essentials of Pharmacology for Health Occupations
[0.   0.   0.13 0.02 0.02 0.01 0.22 0.   0.1  0.49]



In [ ]:
#For step 7, I looked at the topic mixtures for five titles. “Essentials of Human Anatomy & Physiology” was about one thing, with 94% on one topic.
#The other four were mixed. “The Theater Experience” split across three topics (36%, 26%, 22%), and “Essentials of Pharmacology” was 49% on one topic and 22% on another.
#Titles with specific technical words land on one topic, while broader titles spread out because their words fit several topics.